In [1]:
import os
import requests
import pandas as pd 

from dotenv import load_dotenv

In [2]:
load_dotenv()

fred_api_key=os.getenv("FRED_API_KEY")
print(fred_api_key is not None)

True


In [3]:
url="https://api.stlouisfed.org/fred/series/observations"
params = {
    "series_id": "MORTGAGE30US",
    "api_key" : fred_api_key,
    "file_type" : "json"
}
response = requests.get(url,params = params)
response.raise_for_status()
print(response.status_code)

200


In [4]:
data = response.json()
print(type(data))
print(data.keys())

<class 'dict'>
dict_keys(['realtime_start', 'realtime_end', 'observation_start', 'observation_end', 'units', 'output_type', 'file_type', 'order_by', 'sort_order', 'count', 'offset', 'limit', 'observations'])


In [5]:
print(data["observations"][:2])

[{'realtime_start': '2026-08-13', 'realtime_end': '2026-08-13', 'date': '1971-04-02', 'value': '7.33'}, {'realtime_start': '2026-08-13', 'realtime_end': '2026-08-13', 'date': '1971-04-09', 'value': '7.31'}]


In [6]:
df_observations= pd.DataFrame(data["observations"])
df_observations.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2890 entries, 0 to 2889
Data columns (total 4 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   realtime_start  2890 non-null   object
 1   realtime_end    2890 non-null   object
 2   date            2890 non-null   object
 3   value           2890 non-null   object
dtypes: object(4)
memory usage: 90.4+ KB


In [7]:
df_observations.head(10)

,realtime_start,realtime_end,date,value
0,2026-08-13,2026-08-13,1971-04-02,7.33
1,2026-08-13,2026-08-13,1971-04-09,7.31
2,2026-08-13,2026-08-13,1971-04-16,7.31
3,2026-08-13,2026-08-13,1971-04-23,7.31
4,2026-08-13,2026-08-13,1971-04-30,7.29
5,2026-08-13,2026-08-13,1971-05-07,7.38
6,2026-08-13,2026-08-13,1971-05-14,7.42
7,2026-08-13,2026-08-13,1971-05-21,7.44
8,2026-08-13,2026-08-13,1971-05-28,7.46
9,2026-08-13,2026-08-13,1971-06-04,7.52


In [8]:
df_observations=df_observations.rename(columns= {"value" : "mortgage_rate"})

In [9]:
df_observations=df_observations.drop(columns={"realtime_start","realtime_end"})


In [10]:
df_observations.head()

,date,mortgage_rate
0,1971-04-02,7.33
1,1971-04-09,7.31
2,1971-04-16,7.31
3,1971-04-23,7.31
4,1971-04-30,7.29


In [11]:
df_observations["date"]= pd.to_datetime(df_observations["date"])
df_observations["mortgage_rate"]=pd.to_numeric(df_observations["mortgage_rate"])
df_observations.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2890 entries, 0 to 2889
Data columns (total 2 columns):
 #   Column         Non-Null Count  Dtype         
---  ------         --------------  -----         
 0   date           2890 non-null   datetime64[ns]
 1   mortgage_rate  2890 non-null   float64       
dtypes: datetime64[ns](1), float64(1)
memory usage: 45.3 KB


In [12]:
assert df_observations["date"].isnull().sum() == 0
assert df_observations["date"].is_unique

assert df_observations["mortgage_rate"].isnull().sum() == 0
assert (df_observations["mortgage_rate"] > 0).all

In [13]:
df_observations["month"] = df_observations["date"].dt.to_period("M")

In [14]:
df_monthly_rates=(df_observations.groupby("month")["mortgage_rate"].mean())
df_monthly_rates = df_monthly_rates.round(2)

In [15]:
df_monthly_rates.head()

month
1971-04    7.31
1971-05    7.42
1971-06    7.53
1971-07    7.60
1971-08    7.70
Freq: M, Name: mortgage_rate, dtype: float64

In [16]:
df_monthly_rates=pd.DataFrame(df_monthly_rates)
df_monthly_rates = df_monthly_rates.reset_index()

In [17]:
df_monthly_rates.head()

,month,mortgage_rate
0,1971-04,7.31
1,1971-05,7.42
2,1971-06,7.53
3,1971-07,7.60
4,1971-08,7.70


In [18]:
df_monthly_rates.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 665 entries, 0 to 664
Data columns (total 2 columns):
 #   Column         Non-Null Count  Dtype    
---  ------         --------------  -----    
 0   month          665 non-null    period[M]
 1   mortgage_rate  665 non-null    float64  
dtypes: float64(1), period[M](1)
memory usage: 10.5 KB


In [19]:
df_monthly_rates["month"]= df_monthly_rates["month"].dt.to_timestamp()

In [20]:
df_monthly_rates.head()

,month,mortgage_rate
0,1971-04-01,7.31
1,1971-05-01,7.42
2,1971-06-01,7.53
3,1971-07-01,7.60
4,1971-08-01,7.70


In [21]:
assert df_monthly_rates["month"].isnull().sum() == 0 
assert df_monthly_rates["month"].is_unique

assert df_monthly_rates["mortgage_rate"].isnull().sum() == 0
assert (df_monthly_rates["mortgage_rate"] >0).all


In [22]:
import json
with open("../data/raw/fred_mortgage_rate.json", "w") as file:
    json.dump(data, file, indent=4)

In [23]:
df_observations = df_observations.drop(columns=["month"])

In [24]:
df_observations.to_csv("../data/processed/fred_weekly_mortgage_rate.csv",index=False)
df_monthly_rates.to_csv("../data/processed/fred_monthly_mortgage_rate.csv",index=False)